In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/accepted_2007_to_2018Q4.csv", low_memory=False)

In [42]:
df.head()

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,68407277,NaN,3600.0,3600.0,3600.0,36 months,13.99,123.03,C,C4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1,68355089,NaN,24700.0,24700.0,24700.0,36 months,11.99,820.28,C,C1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2,68341763,NaN,20000.0,20000.0,20000.0,60 months,10.78,432.66,B,B4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
3,66310712,NaN,35000.0,35000.0,35000.0,60 months,14.85,829.90,C,C5,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
4,68476807,NaN,10400.0,10400.0,10400.0,60 months,22.45,289.91,F,F1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN


In [43]:
selected_columns = [
    'loan_amnt',
    'term',
    'int_rate',
    'grade',
    'sub_grade',
    'emp_length',
    'home_ownership',
    'annual_inc',
    'verification_status',
    'loan_status',
    'purpose',
    'addr_state',
    'dti',
    'fico_range_low',
    'fico_range_high',
    'installment'
]

df = df[selected_columns]

In [44]:
# Filter valid loan statuses
df = df[df['loan_status'].isin(['Fully Paid', 'Charged Off'])]

In [45]:
df = df.copy()

In [46]:
df.head()

,loan_amnt,term,int_rate,grade,sub_grade,emp_length,home_ownership,annual_inc,verification_status,loan_status,purpose,addr_state,dti,fico_range_low,fico_range_high,installment
0,3600.0,36 months,13.99,C,C4,10+ years,MORTGAGE,55000.0,Not Verified,Fully Paid,debt_consolidation,PA,5.91,675.0,679.0,123.03
1,24700.0,36 months,11.99,C,C1,10+ years,MORTGAGE,65000.0,Not Verified,Fully Paid,small_business,SD,16.06,715.0,719.0,820.28
2,20000.0,60 months,10.78,B,B4,10+ years,MORTGAGE,63000.0,Not Verified,Fully Paid,home_improvement,IL,10.78,695.0,699.0,432.66
4,10400.0,60 months,22.45,F,F1,3 years,MORTGAGE,104433.0,Source Verified,Fully Paid,major_purchase,PA,25.37,695.0,699.0,289.91
5,11950.0,36 months,13.44,C,C3,4 years,RENT,34000.0,Source Verified,Fully Paid,debt_consolidation,GA,10.20,690.0,694.0,405.18


In [47]:
# Create target variable
df['is_default'] = df['loan_status'].apply(lambda x: 1 if x == 'Charged Off' else 0)

In [48]:
df['is_default'].value_counts()

is_default
0    1076751
1     268559
Name: count, dtype: int64

In [49]:
df.head()

,loan_amnt,term,int_rate,grade,sub_grade,emp_length,home_ownership,annual_inc,verification_status,loan_status,purpose,addr_state,dti,fico_range_low,fico_range_high,installment,is_default
0,3600.0,36 months,13.99,C,C4,10+ years,MORTGAGE,55000.0,Not Verified,Fully Paid,debt_consolidation,PA,5.91,675.0,679.0,123.03,0
1,24700.0,36 months,11.99,C,C1,10+ years,MORTGAGE,65000.0,Not Verified,Fully Paid,small_business,SD,16.06,715.0,719.0,820.28,0
2,20000.0,60 months,10.78,B,B4,10+ years,MORTGAGE,63000.0,Not Verified,Fully Paid,home_improvement,IL,10.78,695.0,699.0,432.66,0
4,10400.0,60 months,22.45,F,F1,3 years,MORTGAGE,104433.0,Source Verified,Fully Paid,major_purchase,PA,25.37,695.0,699.0,289.91,0
5,11950.0,36 months,13.44,C,C3,4 years,RENT,34000.0,Source Verified,Fully Paid,debt_consolidation,GA,10.20,690.0,694.0,405.18,0


In [50]:
# Clean emp_length, create function
def clean_emp_lenght(x):
    if pd.isnull(x):
        return None
    elif x == '< 1 year':
        return 0
    elif x =='10+ years':
        return 10
    else:
        return int(x.split()[0])
    
df['emp_length']=df['emp_length'].apply(clean_emp_lenght)

In [51]:
df['emp_length'].value_counts()

emp_length
10.0    442199
2.0     121743
0.0     108061
3.0     107597
1.0      88494
5.0      84154
4.0      80556
6.0      62733
8.0      60701
7.0      59624
9.0      50937
Name: count, dtype: int64

In [52]:
# Handle missing values
df['emp_length'] = df['emp_length'].fillna(df['emp_length'].median())
df['dti'] = df['dti'].fillna(df['dti'].median())

df.head()

,loan_amnt,term,int_rate,grade,sub_grade,emp_length,home_ownership,annual_inc,verification_status,loan_status,purpose,addr_state,dti,fico_range_low,fico_range_high,installment,is_default
0,3600.0,36 months,13.99,C,C4,10.0,MORTGAGE,55000.0,Not Verified,Fully Paid,debt_consolidation,PA,5.91,675.0,679.0,123.03,0
1,24700.0,36 months,11.99,C,C1,10.0,MORTGAGE,65000.0,Not Verified,Fully Paid,small_business,SD,16.06,715.0,719.0,820.28,0
2,20000.0,60 months,10.78,B,B4,10.0,MORTGAGE,63000.0,Not Verified,Fully Paid,home_improvement,IL,10.78,695.0,699.0,432.66,0
4,10400.0,60 months,22.45,F,F1,3.0,MORTGAGE,104433.0,Source Verified,Fully Paid,major_purchase,PA,25.37,695.0,699.0,289.91,0
5,11950.0,36 months,13.44,C,C3,4.0,RENT,34000.0,Source Verified,Fully Paid,debt_consolidation,GA,10.20,690.0,694.0,405.18,0


In [53]:
# Drop remaining null rows (safe now)
df = df.dropna()


In [54]:
df.head()

,loan_amnt,term,int_rate,grade,sub_grade,emp_length,home_ownership,annual_inc,verification_status,loan_status,purpose,addr_state,dti,fico_range_low,fico_range_high,installment,is_default
0,3600.0,36 months,13.99,C,C4,10.0,MORTGAGE,55000.0,Not Verified,Fully Paid,debt_consolidation,PA,5.91,675.0,679.0,123.03,0
1,24700.0,36 months,11.99,C,C1,10.0,MORTGAGE,65000.0,Not Verified,Fully Paid,small_business,SD,16.06,715.0,719.0,820.28,0
2,20000.0,60 months,10.78,B,B4,10.0,MORTGAGE,63000.0,Not Verified,Fully Paid,home_improvement,IL,10.78,695.0,699.0,432.66,0
4,10400.0,60 months,22.45,F,F1,3.0,MORTGAGE,104433.0,Source Verified,Fully Paid,major_purchase,PA,25.37,695.0,699.0,289.91,0
5,11950.0,36 months,13.44,C,C3,4.0,RENT,34000.0,Source Verified,Fully Paid,debt_consolidation,GA,10.20,690.0,694.0,405.18,0


In [57]:
df.shape


(1345310, 17)

In [58]:
df.isnull().sum()

loan_amnt              0
term                   0
int_rate               0
grade                  0
sub_grade              0
emp_length             0
home_ownership         0
annual_inc             0
verification_status    0
loan_status            0
purpose                0
addr_state             0
dti                    0
fico_range_low         0
fico_range_high        0
installment            0
is_default             0
dtype: int64

In [ ]:
# lets make a copy file of the clean data
df_cleaned = df.copy()



Index(['loan_amnt', 'term', 'int_rate', 'grade', 'sub_grade', 'emp_length',
       'home_ownership', 'annual_inc', 'verification_status', 'loan_status',
       'purpose', 'addr_state', 'dti', 'fico_range_low', 'fico_range_high',
       'installment', 'is_default', 'loan_to_income', 'monthly_income',
       'installment_to_income', 'fico_avg', 'risk_segment', 'income_segment'],
      dtype='str')

In [101]:
df_cleaned.to_csv("../data/processed/cleaned_loans.csv", index=False)

In [60]:
# ------ FEATURE ENGINEERING PHASE------- 

#defragment dataframe
df = df.copy()

In [61]:
# Create Financial Ratios - Loan-to-Income Ratio (New Column)

df['loan_to_income'] = df['loan_amnt'] / df['annual_inc']


In [ ]:
# Monthly Income + Installment Burden Ratio

df['monthly_income'] = df['annual_inc'] / 12
df['installment_to_income'] = df['installment'] / df['monthly_income'] 

# 'installment_to_income' = Represents what percentage of the borrower’s monthly income goes to this loan

In [63]:
df.head()

,loan_amnt,term,int_rate,grade,sub_grade,emp_length,home_ownership,annual_inc,verification_status,loan_status,purpose,addr_state,dti,fico_range_low,fico_range_high,installment,is_default,loan_to_income,monthly_income,installment_to_income
0,3600.0,36 months,13.99,C,C4,10.0,MORTGAGE,55000.0,Not Verified,Fully Paid,debt_consolidation,PA,5.91,675.0,679.0,123.03,0,0.065455,4583.333333,0.026843
1,24700.0,36 months,11.99,C,C1,10.0,MORTGAGE,65000.0,Not Verified,Fully Paid,small_business,SD,16.06,715.0,719.0,820.28,0,0.380000,5416.666667,0.151436
2,20000.0,60 months,10.78,B,B4,10.0,MORTGAGE,63000.0,Not Verified,Fully Paid,home_improvement,IL,10.78,695.0,699.0,432.66,0,0.317460,5250.000000,0.082411
4,10400.0,60 months,22.45,F,F1,3.0,MORTGAGE,104433.0,Source Verified,Fully Paid,major_purchase,PA,25.37,695.0,699.0,289.91,0,0.099585,8702.750000,0.033312
5,11950.0,36 months,13.44,C,C3,4.0,RENT,34000.0,Source Verified,Fully Paid,debt_consolidation,GA,10.20,690.0,694.0,405.18,0,0.351471,2833.333333,0.143005


In [ ]:
# Credit Score Feature
# Average FICO

df['fico_avg'] = (df['fico_range_low'] + df['fico_range_high']) / 2

# use to evaluate creditworthiness

In [65]:
# Risk Segmentation - Risk Buckets

def risk_segment(row):
    if row['fico_avg'] > 720 and row['loan_to_income'] < 0.2:
        return 'Low Risk'
    elif row['fico_avg'] > 660 and row['loan_to_income'] < 0.4:
        return 'Medium Risk'
    else:
        return 'High Risk'

df['risk_segment'] = df.apply(risk_segment, axis=1)


In [66]:
df.head()

,loan_amnt,term,int_rate,grade,sub_grade,emp_length,home_ownership,annual_inc,verification_status,loan_status,...,dti,fico_range_low,fico_range_high,installment,is_default,loan_to_income,monthly_income,installment_to_income,fico_avg,risk_segment
0,3600.0,36 months,13.99,C,C4,10.0,MORTGAGE,55000.0,Not Verified,Fully Paid,...,5.91,675.0,679.0,123.03,0,0.065455,4583.333333,0.026843,677.0,Medium Risk
1,24700.0,36 months,11.99,C,C1,10.0,MORTGAGE,65000.0,Not Verified,Fully Paid,...,16.06,715.0,719.0,820.28,0,0.380000,5416.666667,0.151436,717.0,Medium Risk
2,20000.0,60 months,10.78,B,B4,10.0,MORTGAGE,63000.0,Not Verified,Fully Paid,...,10.78,695.0,699.0,432.66,0,0.317460,5250.000000,0.082411,697.0,Medium Risk
4,10400.0,60 months,22.45,F,F1,3.0,MORTGAGE,104433.0,Source Verified,Fully Paid,...,25.37,695.0,699.0,289.91,0,0.099585,8702.750000,0.033312,697.0,Medium Risk
5,11950.0,36 months,13.44,C,C3,4.0,RENT,34000.0,Source Verified,Fully Paid,...,10.20,690.0,694.0,405.18,0,0.351471,2833.333333,0.143005,692.0,Medium Risk


In [ ]:
# Income Segmentation

def income_group(x):
    if x < 40000:
        return 'Low Income'
    elif x < 80000:
        return 'Mid Income'
    else:
        return 'High Income'

df['income_segment'] = df['annual_inc'].apply(income_group)

#Turns raw data (numeric data) into a new column (values) with user-friendly values (Low, Mid, High Income)

In [69]:
df.head()

,loan_amnt,term,int_rate,grade,sub_grade,emp_length,home_ownership,annual_inc,verification_status,loan_status,...,fico_range_low,fico_range_high,installment,is_default,loan_to_income,monthly_income,installment_to_income,fico_avg,risk_segment,income_segment
0,3600.0,36 months,13.99,C,C4,10.0,MORTGAGE,55000.0,Not Verified,Fully Paid,...,675.0,679.0,123.03,0,0.065455,4583.333333,0.026843,677.0,Medium Risk,Mid Income
1,24700.0,36 months,11.99,C,C1,10.0,MORTGAGE,65000.0,Not Verified,Fully Paid,...,715.0,719.0,820.28,0,0.380000,5416.666667,0.151436,717.0,Medium Risk,Mid Income
2,20000.0,60 months,10.78,B,B4,10.0,MORTGAGE,63000.0,Not Verified,Fully Paid,...,695.0,699.0,432.66,0,0.317460,5250.000000,0.082411,697.0,Medium Risk,Mid Income
4,10400.0,60 months,22.45,F,F1,3.0,MORTGAGE,104433.0,Source Verified,Fully Paid,...,695.0,699.0,289.91,0,0.099585,8702.750000,0.033312,697.0,Medium Risk,High Income
5,11950.0,36 months,13.44,C,C3,4.0,RENT,34000.0,Source Verified,Fully Paid,...,690.0,694.0,405.18,0,0.351471,2833.333333,0.143005,692.0,Medium Risk,Low Income


In [71]:
(df['annual_inc'] == 0).sum()

np.int64(361)

In [ ]:
df[['loan_to_income', 'installment_to_income', 'fico_avg']].describe()

#some values gave "inf" or "NaN", the problem is that some values in 'annaul_inc' column = 0 and give a Division by zero problem.
# lets fix that.

s:\0. Data Analyst Portfolio\1.Project 1 - Loan Risk & Approval Optimization Analysis\venv\Lib\site-packages\pandas\core\nanops.py:1027: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)
s:\0. Data Analyst Portfolio\1.Project 1 - Loan Risk & Approval Optimization Analysis\venv\Lib\site-packages\pandas\core\nanops.py:1027: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


,loan_to_income,installment_to_income,fico_avg
count,1.345310e+06,1.345310e+06,1.345310e+06
mean,inf,inf,6.981851e+02
std,NaN,NaN,3.185284e+01
min,1.714286e-04,6.912000e-05,6.270000e+02
25%,1.246903e-01,4.626460e-02,6.720000e+02
50%,2.000000e-01,7.221600e-02,6.920000e+02
75%,2.909091e-01,1.054349e-01,7.120000e+02
max,inf,inf,8.475000e+02


In [72]:
# Lets remove invalida rows:
df = df[df['annual_inc'] > 0]

In [73]:
(df['annual_inc'] == 0).sum()

np.int64(0)

In [75]:
# Now we Recalculate features

df['loan_to_income'] = df['loan_amnt'] / df['annual_inc']

df['monthly_income'] = df['annual_inc'] / 12
df['installment_to_income'] = df['installment'] / df['monthly_income']

In [77]:
df.head()

,loan_amnt,term,int_rate,grade,sub_grade,emp_length,home_ownership,annual_inc,verification_status,loan_status,...,fico_range_low,fico_range_high,installment,is_default,loan_to_income,monthly_income,installment_to_income,fico_avg,risk_segment,income_segment
0,3600.0,36 months,13.99,C,C4,10.0,MORTGAGE,55000.0,Not Verified,Fully Paid,...,675.0,679.0,123.03,0,0.065455,4583.333333,0.026843,677.0,Medium Risk,Mid Income
1,24700.0,36 months,11.99,C,C1,10.0,MORTGAGE,65000.0,Not Verified,Fully Paid,...,715.0,719.0,820.28,0,0.380000,5416.666667,0.151436,717.0,Medium Risk,Mid Income
2,20000.0,60 months,10.78,B,B4,10.0,MORTGAGE,63000.0,Not Verified,Fully Paid,...,695.0,699.0,432.66,0,0.317460,5250.000000,0.082411,697.0,Medium Risk,Mid Income
4,10400.0,60 months,22.45,F,F1,3.0,MORTGAGE,104433.0,Source Verified,Fully Paid,...,695.0,699.0,289.91,0,0.099585,8702.750000,0.033312,697.0,Medium Risk,High Income
5,11950.0,36 months,13.44,C,C3,4.0,RENT,34000.0,Source Verified,Fully Paid,...,690.0,694.0,405.18,0,0.351471,2833.333333,0.143005,692.0,Medium Risk,Low Income


In [78]:
# Just in case let's replace any remaining infinities

import numpy as np

df.replace([np.inf, -np.inf], np.nan, inplace=True)
df = df.dropna()

In [79]:
df[['loan_to_income', 'installment_to_income', 'fico_avg']].describe()

,loan_to_income,installment_to_income,fico_avg
count,1.344949e+06,1.344949e+06,1.344949e+06
mean,3.973574e-01,1.423711e-01,6.981821e+02
std,7.082870e+01,2.364390e+01,3.185101e+01
min,1.714286e-04,6.912000e-05,6.270000e+02
25%,1.246631e-01,4.626102e-02,6.720000e+02
50%,2.000000e-01,7.219970e-02,6.920000e+02
75%,2.909091e-01,1.054020e-01,7.120000e+02
max,4.000000e+04,1.320792e+04,8.475000e+02


In [80]:
df['risk_segment'].value_counts()

risk_segment
Medium Risk    1097393
Low Risk        141673
High Risk       105883
Name: count, dtype: int64

In [83]:
# time to handle outliers: We don’t delete everything, we cap intelligently

df = df[df['loan_to_income'] < 5]
df = df[df['installment_to_income'] < 1]

In [85]:
df[['loan_to_income', 'installment_to_income', 'fico_avg']].describe()

,loan_to_income,installment_to_income,fico_avg
count,1.344690e+06,1.344690e+06,1.344690e+06
mean,2.150954e-01,7.936377e-02,6.981807e+02
std,1.239318e-01,4.585539e-02,3.184983e+01
min,1.714286e-04,6.912000e-05,6.270000e+02
25%,1.246106e-01,4.625760e-02,6.720000e+02
50%,2.000000e-01,7.219286e-02,6.920000e+02
75%,2.909091e-01,1.053663e-01,7.120000e+02
max,3.600000e+00,9.998480e-01,8.475000e+02


In [87]:
# Default Rate by Risk Segment

df.groupby('risk_segment')['is_default'].mean().sort_values(ascending=False)

risk_segment
High Risk      0.304486
Medium Risk    0.204784
Low Risk       0.081378
Name: is_default, dtype: float64

In [88]:
# Income Segment Risk

df.groupby('income_segment')['is_default'].mean()



income_segment
High Income    0.168956
Low Income     0.237492
Mid Income     0.209426
Name: is_default, dtype: float64

In [89]:
#Combine Segments

df.groupby(['income_segment', 'risk_segment'])['is_default'].mean()

income_segment  risk_segment
High Income     High Risk       0.289896
                Low Risk        0.072512
                Medium Risk     0.184572
Low Income      High Risk       0.306966
                Low Risk        0.120509
                Medium Risk     0.233568
Mid Income      High Risk       0.304880
                Low Risk        0.084000
                Medium Risk     0.210596
Name: is_default, dtype: float64

In [90]:
# ------- Save current dataset--------

df.to_csv("../data/processed/feature_engineered_loans.csv", index=False)

In [95]:
df.to_parquet("../data/processed/feature_engineered_loans.parquet", index=False)

ArrowKeyError: A type extension with name pandas.period already defined

In [96]:
df.sample(10000).to_csv("../data/processed/sample_loans.csv", index=False)